In [ ]:
!pip install -q scgpt scanpy gdown langgraph openai psycopg2-binary langchain-core


In [ ]:
import json
import os
import re
from types import ModuleType
from typing import Any, Dict, List, Optional, TypedDict

import gdown
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from langgraph.graph import END, StateGraph
from langchain_core.tools import tool
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
import psycopg2
from psycopg2.extras import RealDictCursor

DATASET_FOLDER_URL = "https://drive.google.com/drive/folders/1OjwOq4DtShEci0jBgF7FpXsH07l21k4k"
MODEL_FOLDER_URL = "https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"
DATA_DIR = "/content/yan_dataset"
MODEL_DIR = "/content/scgpt_ckpt"
CONFIDENCE_THRESHOLD = 0.80

# Put your own DB URL here or set it as env var before running.
DB_URL = os.getenv("DB_URL", "")
SPECIES = "Human"

_ACTIVE_ADATA: Any = None


In [ ]:
# =========================
# 2) Full pipeline code
# =========================
import json
import os
import re
from types import ModuleType
from typing import Any, Dict, List, Optional, TypedDict

import gdown
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from langgraph.graph import END, StateGraph
from langchain_core.tools import tool
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
import psycopg2
from psycopg2.extras import RealDictCursor

DATASET_FOLDER_URL = "https://drive.google.com/drive/folders/1OjwOq4DtShEci0jBgF7FpXsH07l21k4k"
MODEL_FOLDER_URL = "https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"

DATA_DIR = "/content/yan_dataset"
MODEL_DIR = "/content/scgpt_ckpt"
CONFIDENCE_THRESHOLD = 0.80
SPECIES = "Human"

# Set these before run
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["DB_URL"] = "postgresql://..."
DB_URL = os.getenv("DB_URL", "")
MAX_TEACHER_CELLS = 200  # set None to disable cap

_ACTIVE_ADATA: Any = None


def patch_torchtext_compat() -> None:
    # ── STEP 1: Wipe all cached scgpt and torchtext modules ──
    import sys

    for key in list(sys.modules.keys()):
        if "scgpt" in key or "torchtext" in key:
            del sys.modules[key]

    # ── STEP 2: Build and inject the fake torchtext ──
    from types import ModuleType

    class _Vocab:
        def __init__(self, vocab=None):
            self._stoi = dict(vocab) if isinstance(vocab, dict) else {}
            self._itos = list(self._stoi.keys())
            self._default_index = None

        def __len__(self):
            return len(self._stoi)

        def __contains__(self, token):
            return token in self._stoi

        def get_stoi(self):
            return self._stoi

        def get_itos(self):
            return self._itos

        def __getitem__(self, token):
            return self._stoi.get(
                token,
                self._default_index if self._default_index is not None else 0
            )

        def __call__(self, tokens):
            if isinstance(tokens, list):
                return [self._stoi.get(t, self._default_index or 0) for t in tokens]
            return self._stoi.get(tokens, self._default_index or 0)

        def set_default_index(self, idx):
            self._default_index = idx

        def get_default_index(self):
            return self._default_index

        def append_token(self, token):
            if token not in self._stoi:
                self._stoi[token] = len(self._itos)
                self._itos.append(token)

        def insert_token(self, token, index):
            if token not in self._stoi:
                self._itos.insert(index, token)
                self._stoi = {t: i for i, t in enumerate(self._itos)}

    class _VocabBuilt(_Vocab):
        def __init__(self, ordered_dict, min_freq=1):
            super().__init__()
            for token, freq in ordered_dict.items():
                if freq >= min_freq:
                    self._stoi[token] = len(self._itos)
                    self._itos.append(token)
            self.vocab = self._stoi

    def _vocab_builder(ordered_dict, min_freq=1, **kwargs):
        return _VocabBuilt(ordered_dict, min_freq)

    _torchtext = ModuleType("torchtext")
    _torchtext_vocab = ModuleType("torchtext.vocab")
    _torchtext_vocab.Vocab = _Vocab
    _torchtext_vocab.vocab = _vocab_builder
    _torchtext.vocab = _torchtext_vocab

    sys.modules["torchtext"] = _torchtext
    sys.modules["torchtext.vocab"] = _torchtext_vocab

    # ── STEP 3: Verify it worked ──
    import torchtext.vocab as test
    print("Vocab class :", test.Vocab)
    print("vocab fn    :", test.vocab)
    print("✅ Patch applied successfully")


def set_active_adata(adata: Any) -> None:
    global _ACTIVE_ADATA
    _ACTIVE_ADATA = adata


@tool
def get_top_expressed_genes(cell_index: int, top_n: int = 10) -> List[str]:
    """Returns top-N expressed genes for one cell from active AnnData."""
    global _ACTIVE_ADATA
    if _ACTIVE_ADATA is None:
        return []
    try:
        row = _ACTIVE_ADATA[cell_index].X
        row = row.toarray().reshape(-1) if hasattr(row, "toarray") else np.asarray(row).reshape(-1)
        top_idx = np.argsort(row)[::-1][:top_n]
        return _ACTIVE_ADATA.var_names[top_idx].tolist()
    except Exception:
        return []


@tool
def query_cellmarker_db(genes: List[str], species: str = "Human") -> List[Dict[str, Any]]:
    """Looks up likely cell types by marker overlap in CellMarker DB."""
    if not DB_URL:
        return []
    cleaned = [g.strip().upper() for g in genes if isinstance(g, str) and g.strip()]
    if not cleaned:
        return []
    query = """
        SELECT
            INITCAP(REPLACE(cell_name, '-', ' ')) AS standardized_cell_name,
            COUNT(*) AS marker_count
        FROM cell_markers
        WHERE UPPER(marker) = ANY(%s)
          AND LOWER(species) = LOWER(%s)
        GROUP BY standardized_cell_name
        ORDER BY marker_count DESC
        LIMIT 5;
    """
    try:
        with psycopg2.connect(DB_URL) as conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(query, (cleaned, species))
                return list(cur.fetchall())
    except Exception:
        return []


def parse_json_block(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None


def teacher_agent_decision(payload: Dict[str, Any]) -> Dict[str, Any]:
    api_key = os.getenv("OPENAI_API_KEY", "")
    if api_key:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=api_key, timeout=45)
            prompt = {
                "task": "Cell type adjudication for one low-confidence cell",
                "instructions": [
                    "Choose a final cell label.",
                    "Prefer marker evidence from CellMarker lookup.",
                    "Return only JSON with keys: teacher_label, teacher_confidence, reason, status.",
                    "status must be 'ok'."
                ],
                "input": payload,
            }
            resp = client.responses.create(
                model="gpt-5.4-mini",
                input=json.dumps(prompt),
                temperature=0
            )
            data = parse_json_block(resp.output_text)
            if data and all(k in data for k in ["teacher_label", "teacher_confidence", "reason", "status"]):
                return data
        except Exception:
            pass

    hits = payload.get("cellmarker_hits", [])
    if hits:
        best = hits[0]
        return {
            "teacher_label": best.get("standardized_cell_name", payload["pred_label"]),
            "teacher_confidence": min(0.95, max(0.55, 0.55 + 0.05 * float(best.get("marker_count", 0)))),
            "reason": "Heuristic fallback using top CellMarker hit.",
            "status": "ok",
        }
    return {
        "teacher_label": payload["pred_label"],
        "teacher_confidence": payload["pred_confidence"],
        "reason": "No marker evidence; fallback to model prediction.",
        "status": "failed",
    }


class PipelineState(TypedDict, total=False):
    dataset_dir: str
    model_dir: str
    threshold: float
    species: str
    adata: Any
    embeddings: np.ndarray
    labels: np.ndarray
    label_to_idx: Dict[str, int]
    idx_to_label: Dict[int, str]
    pred_labels: List[str]
    hybrid_pred_labels: List[str]
    confidence_scores: np.ndarray
    confidence_gate: List[str]
    high_conf_count: int
    low_conf_count: int
    test_indices: np.ndarray
    y_test_idx: np.ndarray
    low_positions: List[int]
    low_cursor: int
    current_cell: Dict[str, Any]
    teacher_results: Dict[int, Dict[str, Any]]
    teacher_failures: int
    metrics_raw: Dict[str, float]
    metrics_hybrid: Dict[str, float]
    metrics_improvement: Dict[str, float]


class CellTypeMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def _download_folder_if_missing(url: str, output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)
    if os.listdir(output_dir):
        return
    gdown.download_folder(url=url, output=output_dir, quiet=False, use_cookies=False)


def raw_dataset_node(state: PipelineState) -> PipelineState:
    dataset_dir = state.get("dataset_dir", DATA_DIR)
    model_dir = state.get("model_dir", MODEL_DIR)
    _download_folder_if_missing(DATASET_FOLDER_URL, dataset_dir)
    _download_folder_if_missing(MODEL_FOLDER_URL, model_dir)
    return {"dataset_dir": dataset_dir, "model_dir": model_dir}


def dataset_processing_node(state: PipelineState) -> PipelineState:
    data = pd.read_csv(os.path.join(state["dataset_dir"], "yan_data.csv"))
    cell = pd.read_csv(os.path.join(state["dataset_dir"], "yan_celldata.csv"))

    data.index = data.iloc[:, 0]
    data = data.iloc[:, 1:]
    x_df = data.T.reset_index().rename(columns={"index": "cell_id"})
    df = x_df.merge(cell, left_on="cell_id", right_on="Unnamed: 0").drop(columns=["Unnamed: 0"])
    x = df.drop(columns=["cell_id", "cell_type1", "cell_type2"])
    y = df["cell_type1"]

    adata = sc.AnnData(x)
    adata.var_names = x.columns.astype(str)
    adata.obs["cell_type"] = y.values
    return {"adata": adata}


def scgpt_node(state: PipelineState) -> PipelineState:
    patch_torchtext_compat()
    from scgpt.tasks import embed_data

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    adata = embed_data(
        state["adata"],
        model_dir=state["model_dir"],
        gene_col="index",
        max_length=1200,
        batch_size=64,
        device=device,
        use_fast_transformer=False,
        return_new_adata=False,
    )
    set_active_adata(adata)
    embeddings = np.asarray(adata.obsm["X_scGPT"], dtype=np.float32)
    labels = adata.obs["cell_type"].values
    return {"adata": adata, "embeddings": embeddings, "labels": labels}


def prediction_node(state: PipelineState) -> PipelineState:
    embeddings = state["embeddings"]
    labels = state["labels"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    classes = sorted(np.unique(labels))
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}

    x_np = np.asarray(embeddings, dtype=np.float32)
    y_np = np.asarray([label_to_idx[l] for l in labels], dtype=np.int64)
    idx_np = np.arange(len(x_np))

    x_train, x_test, y_train, y_test, _idx_train, idx_test = train_test_split(
        x_np, y_np, idx_np, test_size=0.2, random_state=42, stratify=y_np
    )

    x_train_t = torch.tensor(x_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)
    x_test_t = torch.tensor(x_test, dtype=torch.float32, device=device)

    model = CellTypeMLP(input_dim=x_train_t.shape[1], hidden_dim=256, num_classes=len(classes)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for _ in range(200):
        optimizer.zero_grad()
        logits = model(x_train_t)
        loss = F.cross_entropy(logits, y_train_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits_test = model(x_test_t)
        probs_test = torch.softmax(logits_test, dim=1)
        pred_idx = torch.argmax(probs_test, dim=1).cpu().numpy()
        conf = torch.max(probs_test, dim=1).values.cpu().numpy()

    pred_labels = [idx_to_label[int(i)] for i in pred_idx]
    for i in range(min(5, len(pred_labels))):
        print(i, pred_labels[i], float(conf[i]))

    return {
        "pred_labels": pred_labels,
        "hybrid_pred_labels": pred_labels.copy(),
        "confidence_scores": conf,
        "test_indices": idx_test,
        "y_test_idx": y_test,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
    }


def confidence_check_node(state: PipelineState) -> PipelineState:
    scores = np.asarray(state["confidence_scores"], dtype=float)
    threshold = float(state.get("threshold", CONFIDENCE_THRESHOLD))
    gate = ["High" if s >= threshold else "Low" for s in scores]
    high_count = sum(g == "High" for g in gate)
    low_count = len(gate) - high_count
    print(f"Confidence check complete: High={high_count}, Low={low_count}, threshold={threshold}")
    return {
        "confidence_gate": gate,
        "high_conf_count": high_count,
        "low_conf_count": low_count,
        "threshold": threshold,
    }


def init_teacher_queue_node(state: PipelineState) -> PipelineState:
    low_positions = [i for i, g in enumerate(state["confidence_gate"]) if g == "Low"]
    if MAX_TEACHER_CELLS is not None:
        low_positions = low_positions[:MAX_TEACHER_CELLS]
    print(f"[InitTeacherQueue] total_low_for_teacher={len(low_positions)}")
    return {
        "low_positions": low_positions,
        "low_cursor": 0,
        "teacher_results": {},
        "teacher_failures": 0,
    }


def route_teacher_loop(state: PipelineState) -> str:
    cursor = int(state["low_cursor"])
    total = len(state["low_positions"])
    if cursor < total:
        print(f"[TeacherLoop] progress={cursor}/{total}")
        return "prepare_teacher_cell"
    print(f"[TeacherLoop] completed {total}/{total}")
    return "metrics"


def prepare_teacher_cell_node(state: PipelineState) -> PipelineState:
    pos = state["low_positions"][state["low_cursor"]]
    adata_idx = int(state["test_indices"][pos])
    genes = get_top_expressed_genes.invoke({"cell_index": adata_idx, "top_n": 20})
    hits = query_cellmarker_db.invoke({"genes": genes, "species": state.get("species", SPECIES)})

    current = {
        "position": pos,
        "adata_index": adata_idx,
        "pred_label": state["pred_labels"][pos],
        "pred_confidence": float(state["confidence_scores"][pos]),
        "top_genes": genes,
        "cellmarker_hits": hits,
    }

    print(
        f"[PrepareTeacherCell] pos={pos} adata_idx={adata_idx} "
        f"pred={current['pred_label']} conf={current['pred_confidence']:.4f} "
        f"genes={len(genes)} hits={len(hits)}"
    )
    return {"current_cell": current}


def teacher_agent_node(state: PipelineState) -> PipelineState:
    current = state["current_cell"]
    decision = teacher_agent_decision(current)
    out = dict(state["teacher_results"])
    out[current["position"]] = decision

    print(
        f"[TeacherAgent] pos={current['position']} "
        f"label={decision.get('teacher_label')} "
        f"t_conf={float(decision.get('teacher_confidence', 0.0)):.4f} "
        f"status={decision.get('status')}"
    )
    return {"teacher_results": out}


def merge_teacher_result_node(state: PipelineState) -> PipelineState:
    current = state["current_cell"]
    pos = current["position"]
    decision = state["teacher_results"][pos]
    hybrid = list(state["hybrid_pred_labels"])
    failures = int(state["teacher_failures"])

    if decision.get("status") == "ok":
        hybrid[pos] = str(decision.get("teacher_label", current["pred_label"]))
        source = "teacher"
    else:
        failures += 1
        source = "fallback_model"

    next_cursor = int(state["low_cursor"]) + 1
    print(f"[Merge] pos={pos} final={hybrid[pos]} source={source} next_cursor={next_cursor}")

    return {
        "hybrid_pred_labels": hybrid,
        "teacher_failures": failures,
        "low_cursor": next_cursor,
    }


def _metric_pack(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    p_m, r_m, f_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p_m),
        "recall_macro": float(r_m),
        "f1_macro": float(f_m),
        "precision_weighted": float(p_w),
        "recall_weighted": float(r_w),
        "f1_weighted": float(f_w),
    }


def metrics_node(state: PipelineState) -> PipelineState:
    l2i = state["label_to_idx"]
    y_true = np.asarray(state["y_test_idx"], dtype=int)
    y_raw = np.asarray([l2i[x] for x in state["pred_labels"]], dtype=int)
    y_hybrid = np.asarray([l2i.get(x, l2i[state["pred_labels"][i]]) for i, x in enumerate(state["hybrid_pred_labels"])], dtype=int)

    raw = _metric_pack(y_true, y_raw)
    hybrid = _metric_pack(y_true, y_hybrid)
    improvement = {k: hybrid[k] - raw[k] for k in raw.keys()}

    print("\n=== Evaluation Report (Test Set) ===")
    print(f"Total test cells: {len(y_true)}")
    print(f"Low-confidence cells sent to teacher: {len(state['low_positions'])}")
    print(f"Teacher failed (fallback to model): {state['teacher_failures']}")

    print("\n--- Raw Model (scGPT only) ---")
    for k, v in raw.items():
        print(f"{k:18}: {v:.3f}")

    print("\n--- Hybrid Pipeline (scGPT + Teacher) ---")
    for k, v in hybrid.items():
        print(f"{k:18}: {v:.3f}")

    print("\n--- Improvement (Hybrid - Raw) ---")
    for k, v in improvement.items():
        print(f"{k:18}: {v:+.3f}")

    return {
        "metrics_raw": raw,
        "metrics_hybrid": hybrid,
        "metrics_improvement": improvement,
    }


def build_graph():
    graph = StateGraph(PipelineState)
    graph.add_node("raw_dataset", raw_dataset_node)
    graph.add_node("dataset_processing", dataset_processing_node)
    graph.add_node("scgpt", scgpt_node)
    graph.add_node("prediction", prediction_node)
    graph.add_node("confidence_check", confidence_check_node)
    graph.add_node("init_teacher_queue", init_teacher_queue_node)
    graph.add_node("prepare_teacher_cell", prepare_teacher_cell_node)
    graph.add_node("teacher_agent", teacher_agent_node)
    graph.add_node("merge_teacher", merge_teacher_result_node)
    graph.add_node("metrics", metrics_node)

    graph.set_entry_point("raw_dataset")
    graph.add_edge("raw_dataset", "dataset_processing")
    graph.add_edge("dataset_processing", "scgpt")
    graph.add_edge("scgpt", "prediction")
    graph.add_edge("prediction", "confidence_check")
    graph.add_edge("confidence_check", "init_teacher_queue")
    graph.add_conditional_edges("init_teacher_queue", route_teacher_loop)
    graph.add_edge("prepare_teacher_cell", "teacher_agent")
    graph.add_edge("teacher_agent", "merge_teacher")
    graph.add_conditional_edges("merge_teacher", route_teacher_loop)
    graph.add_edge("metrics", END)
    return graph.compile()

In [ ]:
def build_graph():
    graph = StateGraph(PipelineState)
    graph.add_node("raw_dataset", raw_dataset_node)
    graph.add_node("dataset_processing", dataset_processing_node)
    graph.add_node("scgpt", scgpt_node)
    graph.add_node("prediction", prediction_node)
    graph.add_node("confidence_check", confidence_check_node)
    graph.add_node("init_teacher_queue", init_teacher_queue_node)
    graph.add_node("prepare_teacher_cell", prepare_teacher_cell_node)
    graph.add_node("teacher_agent", teacher_agent_node)
    graph.add_node("merge_teacher", merge_teacher_result_node)
    graph.add_node("metrics", metrics_node)

    graph.set_entry_point("raw_dataset")
    graph.add_edge("raw_dataset", "dataset_processing")
    graph.add_edge("dataset_processing", "scgpt")
    graph.add_edge("scgpt", "prediction")
    graph.add_edge("prediction", "confidence_check")
    graph.add_edge("confidence_check", "init_teacher_queue")
    graph.add_conditional_edges("init_teacher_queue", route_teacher_loop)
    graph.add_edge("prepare_teacher_cell", "teacher_agent")
    graph.add_edge("teacher_agent", "merge_teacher")
    graph.add_conditional_edges("merge_teacher", route_teacher_loop)
    graph.add_edge("metrics", END)
    return graph.compile()


In [ ]:
app = build_graph()
final_state = app.invoke(
    {
        "dataset_dir": DATA_DIR,
        "model_dir": MODEL_DIR,
        "threshold": CONFIDENCE_THRESHOLD,
        "species": SPECIES,
    }
)

print("\nPipeline finished.")
